# FlowFeat Demo 

This notebook provides a simple demo of **FlowFeat**.

We will:
1. Install dependencies
2. Load a pretrained FlowFeat model via **PyTorch Hub**
3. Run inference on an image
4. Visualize dense decoder features with PCA
5. Explore a similarity map for point-based retrieval

**Notes**
- First run will download model weights.
- GPU is recommended; CPU works but may be slow.

## 1) Setup

In [ ]:
# If you're in Colab, uncomment the next line:
# !pip -q install --upgrade pip

# Minimal dependencies for this demo
# (torch/torchvision are usually preinstalled in Colab; otherwise install appropriate wheels for your system)
!pip -q install opencv-python pillow matplotlib scikit-learn

import os
import math
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

## 2) Load FlowFeat via PyTorch Hub

In [ ]:
# FlowFeat can be loaded directly from PyTorch Hub.
# Supported variants (as listed in the repo README):
#   dino_vits16_yt
#   dino_vitb16_yt
#   dino_vitb16_kt
#   dinov2_vits14_yt
#   dinov2_vitb14_yt
#   dinov2_vitb14_kt

device = "cuda" if torch.cuda.is_available() else "cpu"

model = torch.hub.load(
    "tum-vision/flowfeat",
    "flowfeat",
    name="dinov2_vits14_yt",
    pretrained=True
).to(device).eval()

print("Loaded model on:", device)

## 3) Load an example image (or your own)

In [ ]:
# Option A: Use your own image 
# img = Image.open("/path/to/your_image.jpg").convert("RGB")

# Option B: Download a sample image from the web (requires internet access in your runtime).
import urllib.request

url = "https://farm3.staticflickr.com/2628/3761093143_0233b13a00_z.jpg"
img_path = "example.jpg"
if not os.path.exists(img_path):
    urllib.request.urlretrieve(url, img_path)

img = Image.open(img_path).convert("RGB")
img

In [ ]:
# (FlowFeat works reasonably well on arbitrary sizes, but memory scales with HxW.)

target_size = 448
w, h = img.size
scale = target_size / max(w, h)
new_w, new_h = int(round(w * scale)), int(round(h // 28 * scale) * 28)
img_resized = img.resize((new_w, new_h), Image.BILINEAR)

plt.figure()
plt.imshow(img_resized)
plt.axis("off")
plt.title(f"Input image {img_resized.size}");

## 4) Preprocess and run inference

In [ ]:
import torchvision.transforms as T

to_tensor = T.Compose([
    T.ToTensor(),
    # FlowFeat uses transformer backbones; standard ImageNet normalization is typically safe.
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

x = to_tensor(img_resized).unsqueeze(0).to(device)  # (1,3,H,W)

with torch.no_grad():
    y_enc, y_dec = model(x)

print("encoder features:", tuple(y_enc.shape))
print("decoder features:", tuple(y_dec.shape))

## 5) Visualize dense decoder features with PCA

In [ ]:
# y_dec: (B, C, H, W) where H,W match the input size and C is the FlowFeat feature dim (typically 128)
feat = y_dec[0].detach().float().cpu()          # (C,H,W)
C, H, W = feat.shape
feat_hw_c = feat.permute(1, 2, 0).reshape(-1, C).numpy()  # (H*W, C)

# PCA to 3 components -> RGB visualization
pca = PCA(n_components=3, random_state=0)
rgb = pca.fit_transform(feat_hw_c)              # (H*W,3)

# Normalize to [0,1] for display
rgb = rgb - rgb.min(axis=0, keepdims=True)
rgb = rgb / (rgb.max(axis=0, keepdims=True) + 1e-8)
rgb_img = rgb.reshape(H, W, 3)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(img_resized)
plt.axis("off")
plt.title("Input")

plt.subplot(1, 2, 2)
plt.imshow(rgb_img)
plt.axis("off")
plt.title("FlowFeat decoder features (PCA->RGB)")
plt.tight_layout()

## 6) Point-based similarity map

In [ ]:
# Pick a point (x0,y0) on the image and visualize cosine similarity to all pixels in feature space.
# Change x0,y0 or click coordinate from a plotting backend if you prefer.

x0, y0 = W // 2, H // 2  # center point

feat_norm = torch.nn.functional.normalize(feat, dim=0)   # (C,H,W), normalized over channel
q = feat_norm[:, y0, x0].view(C, 1, 1)                   # (C,1,1)
sim = (feat_norm * q).sum(dim=0).numpy()                 # (H,W) cosine similarity

# Normalize for display
sim_disp = (sim - sim.min()) / (sim.max() - sim.min() + 1e-8)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(img_resized)
plt.scatter([x0], [y0], s=60)
plt.axis("off")
plt.title("Query point")

plt.subplot(1, 2, 2)
plt.imshow(sim_disp)
plt.axis("off")
plt.title("Cosine similarity map")
plt.tight_layout()